# Taller: Internals de un LLM estilo LLaMA

### 3. Grouped Query Attention vs Multi-Head Attention — efecto de n_kv_heads

En este notebook se exploran variantes del mecanismo de atención, comparando
Multi-Head Attention (MHA) con Grouped Query Attention (GQA). Se analiza cómo
compartir keys y values entre múltiples cabezas reduce el número de parámetros
y el costo computacional. Además, se inspeccionan los mapas de atención para
entender las diferencias en el comportamiento entre ambas configuraciones.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional

# Importamos el modelo de referencia para comparaciones
from src.model import (
    ModelConfig, RMSNorm, SwiGLUFFN, MiniLLaMA,
    precompute_rope_freqs, apply_rope, GroupedQueryAttention
)
from src.data import get_corpus
from src.tokenizer import BPETokenizer

In [2]:
# ===========================================================================
# SECCIÓN 3 — GQA vs MHA
# ===========================================================================
# Multi-Head Attention (MHA): n_kv_heads == n_heads  (sin compartir)
# Grouped Query Attention (GQA): n_kv_heads < n_heads (KV compartidos)
#
# Experimenta cambiando n_kv_heads y observa:
#   - Diferencia en parámetros de K y V
#   - Diferencia en los mapas de atención
# ===========================================================================

def build_attention(n_heads: int, n_kv_heads: int, d_model: int = 32) -> GroupedQueryAttention:
    # TODO 3.1 — Construye un GroupedQueryAttention con los parámetros dados
    cfg = ModelConfig(vocab_size=256, d_model=d_model, n_heads=n_heads,
                      n_kv_heads=n_kv_heads, d_ff=d_model * 2, max_seq_len=128)
    return GroupedQueryAttention(cfg)


def _get_attn_weights(module: GroupedQueryAttention, x, rope_freqs, mask):
    B, T, D = x.shape
    Dh = module.head_dim
    q = module.Wq(x).reshape(B, T, module.n_heads,    Dh)
    k = module.Wk(x).reshape(B, T, module.n_kv_heads, Dh)
    q = apply_rope(q, rope_freqs)
    k = apply_rope(k, rope_freqs)
    k = k.unsqueeze(3).expand(B, T, module.n_kv_heads, module.n_rep, Dh).reshape(B, T, module.n_heads, Dh)
    q = q.transpose(1, 2); k = k.transpose(1, 2)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(Dh)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    return F.softmax(scores, dim=-1)


def compare_attention_maps(attn_mha, attn_gqa, x, rope_freqs, mask):
    # TODO 3.2 — Extrae los pesos de atención (scores softmax) de ambos módulos
    attn_weights_mha = _get_attn_weights(attn_mha, x, rope_freqs, mask)
    attn_weights_gqa = _get_attn_weights(attn_gqa, x, rope_freqs, mask)
    return attn_weights_mha, attn_weights_gqa


# ── Verificación 3 ──────────────────────────────────────────────────────────
def verify_section3():
    print("=" * 55)
    print("VERIFICACIÓN 3 — GQA vs MHA")
    print("=" * 55)
    D, Hq, Hkv, T = 32, 4, 2, 10

    mha = build_attention(n_heads=Hq,  n_kv_heads=Hq,  d_model=D)
    gqa = build_attention(n_heads=Hq,  n_kv_heads=Hkv, d_model=D)

    # Contar parámetros K+V
    mha_kv = mha.Wk.weight.numel() + mha.Wv.weight.numel()
    gqa_kv = gqa.Wk.weight.numel() + gqa.Wv.weight.numel()
    print(f"  MHA  K+V params : {mha_kv:,}")
    print(f"  GQA  K+V params : {gqa_kv:,}")
    print(f"  Reducción       : {(1 - gqa_kv/mha_kv)*100:.0f}%  (esperado {(1-Hkv/Hq)*100:.0f}%)")

    # Shapes del forward
    cfg        = ModelConfig(vocab_size=256, d_model=D, n_heads=Hq,
                             n_kv_heads=Hkv, d_ff=64, max_seq_len=T)
    x          = torch.randn(1, T, D)
    rope_freqs = precompute_rope_freqs(D // Hq, T)
    mask       = torch.tril(torch.ones(T, T))

    out_mha = build_attention(Hq, Hq, D)(x, rope_freqs, mask)
    out_gqa = build_attention(Hq, Hkv, D)(x, rope_freqs, mask)
    print(f"  MHA output shape: {list(out_mha.shape)}  (esperado [1, {T}, {D}])")
    print(f"  GQA output shape: {list(out_gqa.shape)}  (esperado [1, {T}, {D}])")

    shapes_ok = (out_mha.shape == out_gqa.shape == torch.Size([1, T, D]))
    kv_ok     = (gqa_kv == mha_kv * Hkv // Hq)
    if shapes_ok and kv_ok:
        print("\n  ✓ Sección 3 correcta")
    else:
        print("\n  ✗ Revisa tu implementación")

In [3]:
verify_section3()

VERIFICACIÓN 3 — GQA vs MHA
  MHA  K+V params : 2,048
  GQA  K+V params : 1,024
  Reducción       : 50%  (esperado 50%)
  MHA output shape: [1, 10, 32]  (esperado [1, 10, 32])
  GQA output shape: [1, 10, 32]  (esperado [1, 10, 32])

  ✓ Sección 3 correcta
